# YouTube lecture notes

Builds readable notes for one video or a whole playlist: the words that were spoken, then the frame that was on screen at that moment.

Time sync is the whole point. A caption is kept once and placed on the frame that was visible at the **middle** of that caption. The next frame starts the next block. Rolling auto-captions, which repeat the same words, are merged before that alignment so a sentence is not printed three times.

No language model. Subtitles come from YouTube. Punctuation is added only for auto-captions that have almost none, using pauses already present in the timestamps.

## What is inside

| Job | This notebook |
| --- | --- |
| Video and playlist| `yt-dlp` |
| Subtitles | Timed VTT cues (`start` and `end`) |
| Frames | `ffmpeg` scene detection, one pass |
| Near-duplicate slides | Color thumbnail difference in Pillow |
| Clock | Timestamp of the decoded frame |
| Punctuation | Pause heuristic, only when captions lack it |
| Notes | Word and a print-friendly HTML page |

`python-docx` is still the Word writer. HTML is added so the same notes open in a browser with a link back to that second of the video.

## Before you run

The setup cell installs `yt-dlp`, `pillow`, `python-docx`, and `imageio-ffmpeg`. The last one ships an `ffmpeg` binary, so a system install is optional. Node.js on `PATH` lets `yt-dlp` see the full set of YouTube formats. If YouTube blocks captions, set `cookies_file` to a Netscape `cookies.txt` exported from your browser. `cookies_from_browser` can read Chrome or Firefox on the same machine. Safari only works when the notebook itself is running on macOS.


## Setup


In [19]:
from __future__ import annotations

import base64
import html
import json
import re
import shutil
import subprocess
import sys
import time
import urllib.parse
from dataclasses import dataclass
from pathlib import Path

def ensure_packages() -> None:
    needed = {
        "yt_dlp": "yt-dlp",
        "PIL": "pillow",
        "docx": "python-docx",
        "imageio_ffmpeg": "imageio-ffmpeg",
    }
    missing = []
    for module, package in needed.items():
        try:
            __import__(module)
        except ModuleNotFoundError:
            missing.append(package)
    if not missing:
        return
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

ensure_packages()

import imageio_ffmpeg
import yt_dlp
from PIL import Image
from docx import Document
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor

HYPERLINK_REL = "http://schemas.openxmlformats.org/officeDocument/2006/relationships/hyperlink"


def ffmpeg_exe() -> str:
    return imageio_ffmpeg.get_ffmpeg_exe()


def node_exe() -> str | None:
    """Path to a Node.js binary, including nvm installs that Jupyter cannot see on PATH."""
    found = shutil.which("node") or shutil.which("nodejs")
    if found:
        return found
    nvm_nodes = sorted((Path.home() / ".nvm" / "versions" / "node").glob("*/bin/node"))
    if nvm_nodes:
        return str(nvm_nodes[-1])
    for candidate in (Path("/opt/homebrew/bin/node"), Path("/usr/local/bin/node")):
        if candidate.is_file():
            return str(candidate)
    return None


print("ffmpeg:", ffmpeg_exe())
_node = node_exe()
if _node:
    print("node:", _node)
else:
    print("node: not found. YouTube's player is JavaScript, so some video formats may be missing until Node or Deno is installed.")


ffmpeg: /usr/local/lib/python3.13/dist-packages/imageio_ffmpeg/binaries/ffmpeg-linux-x86_64-v7.0.2
node: /usr/bin/node


## Settings

`scene_threshold` is ffmpeg's 0–1 scene score. Slide changes are usually well above `0.30`. Lower it toward `0.15` when a build step only changes a small part of the slide.

`duplicate_threshold` is the average per-channel color difference (0–255) between two consecutive frames. At `6`, only near-identical frames are dropped. Raise it toward `15` if animations still leave duplicates.

`max_gap` inserts an extra frame when nothing visual changes, so a long talking-head stretch is not one endless paragraph. Set it to `0` to keep only real scene cuts.

`text_then_image` prints the speech for that interval and then the frame that was up. Set it to `False` to put the frame first.


In [20]:
@dataclass
class Settings:
    max_height: int = 720
    scene_threshold: float = 0.30
    min_scene_gap: float = 1.0
    max_gap: float = 40.0
    duplicate_threshold: float = 6.0
    paragraph_pause: float = 1.15
    text_then_image: bool = True
    keep_video: bool = False
    sub_langs: tuple[str, ...] = ("en",)
    cookies_from_browser: str | None = None
    cookies_file: str | None = None

    def validate(self) -> None:
        if not 0 < self.scene_threshold < 1:
            raise ValueError("scene_threshold must be between 0 and 1")
        if self.min_scene_gap < 0:
            raise ValueError("min_scene_gap must be >= 0")
        if self.max_gap < 0:
            raise ValueError("max_gap must be >= 0")
        if not 0 <= self.duplicate_threshold <= 255:
            raise ValueError("duplicate_threshold must be between 0 and 255")
        if self.max_height < 144:
            raise ValueError("max_height must be at least 144")
        _check_cookies(self)


@dataclass
class Cue:
    start: float
    end: float
    text: str


@dataclass
class FrameShot:
    time: float
    path: Path


@dataclass
class Segment:
    start: float
    end: float
    text: str
    image: Path
    chapter: str | None


## Download

A playlist URL downloads each video on its own, so one failure does not throw away the rest. A normal watch URL stays a single video even if it carries a `list=` parameter.

Captions are a second request. YouTube lists timed-text URLs on the video (manual tracks first, then auto-captions). Those files are downloaded after the video, with a pause and a retry when YouTube answers 429. A manual track is preferred. WebVTT is preferred, then YouTube's json3 caption format.


In [21]:
def is_playlist_url(url: str) -> bool:
    parsed = urllib.parse.urlparse(url.strip())
    if "playlist" in parsed.path:
        return True
    query = urllib.parse.parse_qs(parsed.query)
    return "list" in query and "v" not in query


def _check_cookies(settings: Settings) -> None:
    browser = (settings.cookies_from_browser or "").strip().lower()
    cookies_file = (settings.cookies_file or "").strip()
    if browser and cookies_file:
        raise ValueError("Set cookies_file or cookies_from_browser, not both.")
    if cookies_file:
        path = Path(cookies_file).expanduser()
        if not path.is_file():
            raise ValueError(
                f"cookies_file was not found: {path}. "
                "Upload a Netscape cookies.txt and set cookies_file to that path."
            )
        return
    if not browser:
        return
    supported = {
        "safari": {"darwin"},
        "chrome": {"darwin", "linux", "win32"},
        "chromium": {"darwin", "linux", "win32"},
        "firefox": {"darwin", "linux", "win32"},
        "edge": {"darwin", "linux", "win32"},
        "brave": {"darwin", "linux", "win32"},
    }
    if browser not in supported:
        names = ", ".join(sorted(supported))
        raise ValueError(f"Unknown browser {browser!r}. Use one of: {names}. Or set cookies_file.")
    if sys.platform not in supported[browser]:
        raise ValueError(
            f"{browser} cookies cannot be read on this kernel ({sys.platform}). "
            "Safari's cookie store exists only on macOS, and this notebook is running on Linux. "
            "In the browser where you are logged into YouTube, export a Netscape cookies.txt, "
            "upload that file, set cookies_file to its path, and set cookies_from_browser back to None."
        )


def _ydl_base(settings: Settings) -> dict:
    opts: dict = {
        "quiet": True,
        "no_warnings": False,
        "retries": 5,
        "fragment_retries": 5,
        "extractor_retries": 3,
        "ffmpeg_location": ffmpeg_exe(),
    }
    node = node_exe()
    if node:
        opts["js_runtimes"] = {"node": {"path": node}}
    _check_cookies(settings)
    if (settings.cookies_file or "").strip():
        opts["cookiefile"] = str(Path(settings.cookies_file).expanduser())
    elif (settings.cookies_from_browser or "").strip():
        opts["cookiesfrombrowser"] = (settings.cookies_from_browser.strip().lower(),)
    return opts


def _format_selector(max_height: int) -> str:
    height = int(max_height)
    # Video-only first: notes need pictures, not the audio track.
    return (
        f"bv*[height<={height}][ext=mp4]/"
        f"bv*[height<={height}]/"
        f"b[height<={height}][ext=mp4]/"
        f"b[height<={height}]/"
        "bv*/b"
    )


def _lang_rank(code: str, langs: tuple[str, ...]) -> int | None:
    folded = code.lower().replace("_", "-")
    for index, lang in enumerate(langs):
        wanted = lang.lower().replace("_", "-")
        if folded == wanted or folded.startswith(wanted + "-"):
            return index
    return None


def _caption_formats(formats: list[dict]) -> list[dict]:
    rank = {"vtt": 0, "json3": 1}
    chosen = [fmt for fmt in formats if fmt.get("url") and fmt.get("ext") in rank]
    chosen.sort(key=lambda fmt: rank[fmt["ext"]])
    return chosen


def _best_caption_track(pool: dict, langs: tuple[str, ...]) -> tuple[str, list[dict]] | None:
    best = None
    for code, formats in pool.items():
        if "live_chat" in code:
            continue
        rank = _lang_rank(code, langs)
        if rank is None:
            continue
        usable = _caption_formats(formats)
        if not usable:
            continue
        exact = 0 if code.lower().replace("_", "-") in {lang.lower().replace("_", "-") for lang in langs} else 1
        candidate = (exact, rank, code, usable)
        if best is None or candidate[:2] < best[:2]:
            best = candidate
    if best is None:
        return None
    return best[2], best[3]


def _http_status(exc: Exception) -> int | None:
    for attr in ("status", "code"):
        value = getattr(exc, attr, None)
        if isinstance(value, int):
            return value
    match = re.search(r"HTTP Error (\d+)", str(exc))
    return int(match.group(1)) if match else None


def _read_caption(ydl, url: str, attempts: int = 4) -> str:
    delay = 3.0
    for attempt in range(1, attempts + 1):
        try:
            with ydl.urlopen(url) as response:
                return response.read().decode("utf-8", errors="replace")
        except Exception as exc:
            status = _http_status(exc)
            if status not in (429, 403, 503) or attempt == attempts:
                raise
            print(f"  captions returned HTTP {status}; waiting {delay:.0f}s ({attempt}/{attempts})")
            time.sleep(delay)
            delay = min(delay * 2, 30)
    raise RuntimeError(f"could not download captions from {url}")


def list_sources(url: str, limit: int | None, settings: Settings) -> list[dict]:
    opts = _ydl_base(settings)
    opts["skip_download"] = True
    opts["extract_flat"] = "in_playlist"
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=False)
    entries = info.get("entries") if isinstance(info, dict) else None
    if is_playlist_url(url) and entries:
        videos = []
        for entry in entries:
            if not entry:
                continue
            video_id = entry.get("id")
            page = entry.get("url") or entry.get("webpage_url") or ""
            if page and not str(page).startswith("http"):
                page = f"https://www.youtube.com/watch?v={video_id or page}"
            if not page and video_id:
                page = f"https://www.youtube.com/watch?v={video_id}"
            videos.append({
                "id": video_id,
                "url": page,
                "title": entry.get("title") or video_id or page,
            })
        if limit is not None:
            videos = videos[:limit]
        return videos
    page = info.get("webpage_url") or url
    return [{"id": info.get("id"), "url": page, "title": info.get("title") or page}]


def download_video(url: str, dest: Path, settings: Settings) -> dict:
    """Download the picture track only. Captions are fetched afterwards by save_captions."""
    dest.mkdir(parents=True, exist_ok=True)
    opts = _ydl_base(settings)
    opts.update({
        "noplaylist": True,
        "format": _format_selector(settings.max_height),
        "outtmpl": str(dest / "%(id)s.%(ext)s"),
        "writesubtitles": False,
        "writeautomaticsub": False,
        "writeinfojson": True,
        "overwrites": True,
    })
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=True)
    if info.get("requested_downloads"):
        # Playlist-like wrappers are disabled; still unwrap a single result.
        info = info["requested_downloads"][0] or info
    return info


def save_captions(info: dict, dest: Path, settings: Settings) -> tuple[Path | None, str]:
    """Save one caption file. Manual tracks win over auto-captions; vtt wins over json3."""
    video_id = str(info.get("id") or "")
    if not video_id:
        return None, "none"
    pools = (
        (info.get("subtitles") or {}, "manual"),
        (info.get("automatic_captions") or {}, "auto-generated"),
    )
    tracks = []
    for pool, kind in pools:
        picked = _best_caption_track(pool, settings.sub_langs)
        if picked:
            code, formats = picked
            tracks.append((code, formats, f"{code} ({kind})"))
    if not tracks:
        return None, "none"

    opts = _ydl_base(settings)
    last_status = None
    with yt_dlp.YoutubeDL(opts) as ydl:
        for code, formats, label in tracks:
            for fmt in formats:
                try:
                    raw = _read_caption(ydl, fmt["url"])
                except Exception as exc:
                    last_status = _http_status(exc)
                    print(f"  could not download {label} ({fmt.get('ext')}): {exc}")
                    continue
                path = dest / f"{video_id}.{code}.{fmt['ext']}"
                path.write_text(raw, encoding="utf-8")
                return path, label
        fallback_lang = settings.sub_langs[0] if settings.sub_langs else "en"
        for kind, extra in (("manual", ""), ("auto-generated", "&kind=asr")):
            fallback = f"https://www.youtube.com/api/timedtext?v={video_id}&lang={fallback_lang}{extra}&fmt=vtt"
            try:
                raw = _read_caption(ydl, fallback, attempts=2)
            except Exception as exc:
                last_status = _http_status(exc) or last_status
                continue
            if "-->" not in raw:
                continue
            path = dest / f"{video_id}.{fallback_lang}.vtt"
            path.write_text(raw, encoding="utf-8")
            return path, f"{fallback_lang} ({kind})"
    if last_status == 429:
        print("  captions are still rate-limited (HTTP 429). Set cookies_from_browser to \"safari\", \"chrome\", or \"firefox\" and run this video again.")
    return None, "none"


## Subtitles

YouTube's automatic captions are a rolling window: each cue repeats the previous words and adds a few new ones. The repeated words are dropped. The new words keep the clock of the cue they arrived in, so a whole lecture is not glued into one paragraph.

Those words are then cut by the gaps between frames. The words spoken between frame 9 and frame 10 are printed above frame 10. If that cut lands in the middle of a sentence, the rest of the sentence is moved up so the line is finished before the image. A silence, or a `.` `?` `!`, ends the sentence. The last frame also keeps any speech after it, through the end of the video.


In [22]:
_SOUND = re.compile(r"\[(?:music|applause|laughter|inaudible|noise)\]", re.IGNORECASE)
_TAG = re.compile(r"<[^>]+>")
_CUE_TIME = re.compile(r"<\d{2}:\d{2}:\d{2}[.,]\d{3}>")
_WORD = re.compile(r"\w")


def parse_timestamp(value: str) -> float:
    token = value.strip().split()[0].replace(",", ".")
    parts = token.split(":")
    try:
        numbers = [float(part) for part in parts]
    except ValueError as exc:
        raise ValueError(f"Bad timestamp: {value}") from exc
    if len(numbers) == 3:
        hours, minutes, seconds = numbers
    elif len(numbers) == 2:
        hours = 0.0
        minutes, seconds = numbers
    elif len(numbers) == 1:
        hours = minutes = 0.0
        seconds = numbers[0]
    else:
        raise ValueError(f"Bad timestamp: {value}")
    return hours * 3600 + minutes * 60 + seconds


def clean_cue_text(raw: str) -> str:
    text = html.unescape(raw.replace("\u00a0", " "))
    text = _CUE_TIME.sub(" ", text)
    text = _TAG.sub(" ", text)
    text = text.replace("♪", " ")
    text = _SOUND.sub(" ", text)
    text = text.replace(">>", " ")
    lines = []
    for line in text.splitlines():
        line = re.sub(r"\s+", " ", line).strip()
        if not line:
            continue
        if lines and lines[-1] == line:
            continue
        lines.append(line)
    text = re.sub(r"\s+", " ", " ".join(lines)).strip()
    return text


def parse_vtt_text(raw: str) -> list[Cue]:
    raw = raw.lstrip("\ufeff")
    raw = re.sub(r"(?is)\nSTYLE\b.*?\n\n", "\n\n", raw)
    raw = re.sub(r"(?is)\nNOTE\b.*?\n\n", "\n\n", raw)
    cues: list[Cue] = []
    lines = raw.splitlines()
    index = 0
    while index < len(lines):
        line = lines[index].strip()
        if "-->" not in line:
            index += 1
            continue
        left, right = line.split("-->", 1)
        start = parse_timestamp(left)
        end = parse_timestamp(right)
        if end <= start:
            end = start + 0.01
        index += 1
        body: list[str] = []
        while index < len(lines) and lines[index].strip():
            body.append(lines[index])
            index += 1
        text = clean_cue_text("\n".join(body))
        if text and _WORD.search(text):
            cues.append(Cue(start, end, text))
    return cues


def parse_vtt(path: Path) -> list[Cue]:
    return parse_vtt_text(path.read_text(encoding="utf-8", errors="replace"))


def parse_json3_text(raw: str) -> list[Cue]:
    """YouTube's json3 caption file: each event has tStartMs, dDurationMs, and segs."""
    data = json.loads(raw)
    events = data.get("events") if isinstance(data, dict) else data
    cues: list[Cue] = []
    for event in events or []:
        if not isinstance(event, dict):
            continue
        text = clean_cue_text("".join(str(seg.get("utf8", "")) for seg in (event.get("segs") or [])))
        if not text or not _WORD.search(text):
            continue
        start = float(event.get("tStartMs") or 0) / 1000.0
        duration = float(event.get("dDurationMs") or 0) / 1000.0
        end = start + (duration if duration > 0 else 0.01)
        cues.append(Cue(start, end, text))
    return cues


def _norm_token(token: str) -> str:
    return re.sub(r"^[^\w']+|[^\w']+$", "", token).lower()


def _prefix_overlap(prev_words: list[str], new_words: list[str]) -> int:
    prev_norm = [_norm_token(word) for word in prev_words]
    new_norm = [_norm_token(word) for word in new_words]
    max_k = min(len(prev_norm), len(new_norm))
    for length in range(max_k, 0, -1):
        if prev_norm[-length:] == new_norm[:length] and any(prev_norm[-length:]):
            return length
    return 0


def merge_rolling_cues(cues: list[Cue], join_gap: float = 0.40) -> list[Cue]:
    """Drop repeated auto-caption words. New words keep the cue time they arrived in."""
    kept: list[Cue] = []
    previous_words: list[str] = []
    previous_end: float | None = None
    for cue in cues:
        words = cue.text.split()
        if not words:
            continue
        end = max(cue.end, cue.start + 0.01)
        if previous_end is None or cue.start > previous_end + join_gap:
            kept.append(Cue(cue.start, end, cue.text))
            previous_words = words
            previous_end = end
            continue
        overlap = _prefix_overlap(previous_words, words)
        shorter = min(len(previous_words), len(words))
        shares_phrase = overlap >= 2 or (overlap >= 1 and overlap >= shorter)
        if not shares_phrase:
            kept.append(Cue(cue.start, end, cue.text))
            previous_words = words
            previous_end = end
            continue
        extra = words[overlap:]
        if extra:
            kept.append(Cue(cue.start, end, " ".join(extra)))
        previous_words = words
        previous_end = max(previous_end, end)
    return kept


def prepare_cues(cues: list[Cue]) -> list[Cue]:
    ordered = sorted(
        (Cue(cue.start, max(cue.end, cue.start + 0.01), cue.text.strip()) for cue in cues if cue.text.strip()),
        key=lambda cue: (cue.start, cue.end),
    )
    return merge_rolling_cues(ordered)


def captions_lack_punctuation(cues: list[Cue]) -> bool:
    text = " ".join(cue.text for cue in cues)
    words = text.split()
    if len(words) < 40:
        return False
    marks = sum(text.count(mark) for mark in ".?!")
    return (marks / len(words)) < 0.02


def _capitalize_first(text: str) -> str:
    for index, char in enumerate(text):
        if char.isalpha():
            return text[:index] + char.upper() + text[index + 1:]
    return text


def join_cues(cues: list[Cue], add_periods: bool, paragraph_pause: float) -> str:
    chunks: list[list[str]] = []
    current: list[str] = []
    prev_end: float | None = None
    for cue in cues:
        if prev_end is not None and cue.start - prev_end >= paragraph_pause and current:
            chunks.append(current)
            current = []
        current.append(cue.text)
        prev_end = cue.end
    if current:
        chunks.append(current)
    paragraphs = []
    for chunk in chunks:
        text = re.sub(r"\s+", " ", " ".join(chunk)).strip()
        text = re.sub(r"\s+([,.;:?!])", r"\1", text)
        if not text:
            continue
        if add_periods and text[-1] not in ".?!":
            text += "."
        if add_periods:
            text = _capitalize_first(text)
        paragraphs.append(text)
    return "\n\n".join(paragraphs).strip()


def chapter_at(chapters: list[dict] | None, moment: float) -> str | None:
    if not chapters:
        return None
    for chapter in chapters:
        start = float(chapter.get("start_time") or 0)
        end = float(chapter.get("end_time") or 10**9)
        if start <= moment < end:
            title = str(chapter.get("title") or "").strip()
            return title or None
    return None


def _pieces_between(cues: list[Cue], start: float, end: float) -> list[Cue]:
    """Cues whose words fall in [start, end). A cue that crosses the cut is split."""
    if end <= start:
        return []
    pieces: list[Cue] = []
    for cue in cues:
        if cue.end <= start or cue.start >= end:
            continue
        words = cue.text.split()
        if not words:
            continue
        if cue.start >= start and cue.end <= end:
            pieces.append(cue)
            continue
        span = max(cue.end - cue.start, 1e-6)
        chosen = [
            word
            for index, word in enumerate(words)
            if start <= cue.start + span * (index + 0.5) / len(words) < end
        ]
        if chosen:
            pieces.append(Cue(max(cue.start, start), min(cue.end, end), " ".join(chosen)))
    return pieces


def _ends_sentence(text: str) -> bool:
    stripped = text.strip().rstrip("\"')]")
    return bool(stripped) and stripped[-1] in ".?!"


def _pull_until_sentence(donor: list[Cue], pause: float, words_left: int) -> tuple[list[Cue], list[Cue], int]:
    """Take a leading sentence fragment from the next block. Stop at punctuation or a silence."""
    taken: list[Cue] = []
    used = 0
    for index, cue in enumerate(donor):
        if index > 0 and cue.start - donor[index - 1].end >= pause:
            return taken, donor[index:], used
        words = cue.text.split()
        keep: list[str] = []
        rest: list[str] = []
        closed = False
        for word in words:
            if closed:
                rest.append(word)
                continue
            keep.append(word)
            used += 1
            if _ends_sentence(word) or used >= words_left:
                closed = True
        if keep:
            taken.append(Cue(cue.start, cue.end, " ".join(keep)))
        if rest or closed:
            remain = []
            if rest:
                remain.append(Cue(cue.start, cue.end, " ".join(rest)))
            remain.extend(donor[index + 1:])
            return taken, remain, used
    return taken, [], used


def _finish_sentences(buckets: list[list[Cue]], pause: float, max_words: int = 30) -> None:
    """If a block ends mid-sentence, move the continuation up from the following blocks."""
    for index in range(len(buckets)):
        pulled = 0
        while pulled < max_words:
            text = " ".join(cue.text for cue in buckets[index]).strip()
            if not text or _ends_sentence(text):
                break
            nxt = next((later for later in range(index + 1, len(buckets)) if buckets[later]), None)
            if nxt is None:
                break
            if buckets[nxt][0].start - buckets[index][-1].end >= pause:
                break
            taken, remain, used = _pull_until_sentence(buckets[nxt], pause, max_words - pulled)
            if not taken:
                break
            buckets[index].extend(taken)
            buckets[nxt] = remain
            pulled += used
            if _ends_sentence(taken[-1].text):
                break


def build_segments(
    frames: list[FrameShot],
    cues: list[Cue],
    duration: float,
    chapters: list[dict] | None,
    settings: Settings,
) -> list[Segment]:
    if not frames:
        return []
    frames = sorted(frames, key=lambda frame: frame.time)
    end_time = max(duration, frames[-1].time + 0.01)
    add_periods = captions_lack_punctuation(cues)
    buckets: list[list[Cue]] = []
    windows: list[tuple[float, float]] = []
    last = len(frames) - 1
    for index, frame in enumerate(frames):
        if index == 0:
            window_start, window_end = 0.0, frame.time
        elif index == last:
            window_start, window_end = frames[index - 1].time, end_time
        else:
            window_start, window_end = frames[index - 1].time, frame.time
        buckets.append(_pieces_between(cues, window_start, window_end))
        if window_end <= window_start:
            window_start = window_end = frame.time
        windows.append((window_start, window_end))
    _finish_sentences(buckets, settings.paragraph_pause)
    segments: list[Segment] = []
    for frame, bucket, (window_start, window_end) in zip(frames, buckets, windows):
        if bucket:
            window_end = max(window_end, bucket[-1].end)
        segments.append(Segment(
            start=window_start,
            end=window_end,
            text=join_cues(bucket, add_periods, settings.paragraph_pause),
            image=frame.path,
            chapter=chapter_at(chapters, frame.time),
        ))
    return segments


## Frames

Scene detection keeps a frame when the picture actually changes, including the opening frame. Consecutive frames closer than `min_scene_gap`, and consecutive frames whose colors almost match, are dropped. The earlier timestamp is the one that stays, because that is when the slide appeared.

Long gaps are filled by a fast seek. Those extra frames are kept even when the picture matches, so the transcript still breaks into readable blocks.


In [23]:
def fmt_ts(seconds: float) -> str:
    total = max(0, int(seconds))
    hours, rem = divmod(total, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}:{minutes:02d}:{secs:02d}"
    return f"{minutes}:{secs:02d}"


def frame_name(moment: float) -> str:
    return f"t_{int(round(moment * 1000)):09d}.jpg"


def color_distance(left: Image.Image, right: Image.Image) -> float:
    """Mean absolute RGB difference on a small thumbnail. 0 is the same picture."""
    resample = getattr(getattr(Image, "Resampling", Image), "LANCZOS")
    width, height = 32, 18
    a = list(left.convert("RGB").resize((width, height), resample).getdata())
    b = list(right.convert("RGB").resize((width, height), resample).getdata())
    total = 0
    for (r1, g1, b1), (r2, g2, b2) in zip(a, b):
        total += abs(r1 - r2) + abs(g1 - g2) + abs(b1 - b2)
    return total / (len(a) * 3)


def probe_duration(video: Path) -> float:
    proc = subprocess.run(
        [ffmpeg_exe(), "-hide_banner", "-i", str(video)],
        capture_output=True,
        text=True,
    )
    match = re.search(r"Duration:\s*(\d+):(\d+):(\d+(?:\.\d+)?)", proc.stderr)
    if not match:
        raise RuntimeError(f"Could not read duration of {video.name}")
    hours, minutes, seconds = match.groups()
    return int(hours) * 3600 + int(minutes) * 60 + float(seconds)


def _grab_frame(video: Path, moment: float, dest: Path) -> bool:
    dest.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        ffmpeg_exe(), "-hide_banner", "-loglevel", "error",
        "-ss", f"{max(moment, 0):.3f}",
        "-i", str(video),
        "-frames:v", "1",
        "-q:v", "4",
        "-y",
        str(dest),
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    return proc.returncode == 0 and dest.exists() and dest.stat().st_size > 0


def _scene_shots(video: Path, raw_dir: Path, threshold: float) -> list[FrameShot]:
    raw_dir.mkdir(parents=True, exist_ok=True)
    pattern = str(raw_dir / "scene_%06d.jpg")
    select = f"select='eq(n\\,0)+gt(scene\\,{threshold:.3f})',showinfo"
    cmd = [
        ffmpeg_exe(), "-hide_banner", "-loglevel", "info",
        "-an", "-sn",
        "-i", str(video),
        "-vf", select,
        "-fps_mode", "vfr",
        "-q:v", "4",
        pattern,
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        tail = (proc.stderr or "")[-2000:]
        raise RuntimeError(f"Scene detection failed for {video.name}\n{tail}")
    times = [float(value) for value in re.findall(r"pts_time:([0-9]+(?:\.[0-9]+)?)", proc.stderr)]
    files = sorted(raw_dir.glob("scene_*.jpg"))
    count = min(len(times), len(files))
    if len(times) != len(files):
        print(f"  frame/timestamp mismatch ({len(files)} files, {len(times)} times); keeping {count}")
    shots = [FrameShot(times[i], files[i]) for i in range(count)]
    used = {shot.path for shot in shots}
    for extra in files:
        if extra not in used:
            extra.unlink(missing_ok=True)
    return shots


def _drop_close(shots: list[FrameShot], min_gap: float) -> list[FrameShot]:
    kept: list[FrameShot] = []
    for shot in shots:
        if kept and shot.time - kept[-1].time < min_gap:
            shot.path.unlink(missing_ok=True)
            continue
        kept.append(shot)
    return kept


def _drop_similar(shots: list[FrameShot], max_distance: float) -> list[FrameShot]:
    kept: list[FrameShot] = []
    previous: Image.Image | None = None
    for shot in shots:
        try:
            image = Image.open(shot.path).convert("RGB")
        except Exception:
            kept.append(shot)
            previous = None
            continue
        if previous is not None and color_distance(previous, image) <= max_distance:
            image.close()
            shot.path.unlink(missing_ok=True)
            continue
        if previous is not None:
            previous.close()
        kept.append(shot)
        previous = image
    if previous is not None:
        previous.close()
    return kept


def _fill_gaps(video: Path, shots: list[FrameShot], duration: float, max_gap: float, raw_dir: Path) -> list[FrameShot]:
    if max_gap <= 0 or duration <= 0:
        return shots
    extra: list[FrameShot] = []
    points = [shot.time for shot in shots] + [duration]
    for index in range(len(points) - 1):
        moment = points[index] + max_gap
        while moment < points[index + 1] - 0.25 and moment < duration - 0.05:
            dest = raw_dir / f"gap_{int(round(moment * 1000)):09d}.jpg"
            if _grab_frame(video, moment, dest):
                extra.append(FrameShot(moment, dest))
            moment += max_gap
    return sorted(shots + extra, key=lambda shot: shot.time)


def extract_frames(video: Path, dest: Path, settings: Settings, duration: float) -> list[FrameShot]:
    dest.mkdir(parents=True, exist_ok=True)
    raw_dir = dest / "_raw"
    if raw_dir.exists():
        shutil.rmtree(raw_dir)
    raw_dir.mkdir()
    shots = _scene_shots(video, raw_dir, settings.scene_threshold)
    raw_count = len(shots)
    shots = _drop_close(shots, settings.min_scene_gap)
    shots = _drop_similar(shots, settings.duplicate_threshold)
    if not shots:
        opening = raw_dir / "fallback.jpg"
        if _grab_frame(video, 0.0, opening):
            shots = [FrameShot(0.0, opening)]
    shots = _fill_gaps(video, shots, duration, settings.max_gap, raw_dir)
    final: list[FrameShot] = []
    for shot in shots:
        target = dest / frame_name(shot.time)
        if target.exists():
            target = dest / f"{target.stem}_{len(final)}{target.suffix}"
        shutil.move(str(shot.path), target)
        final.append(FrameShot(shot.time, target))
    shutil.rmtree(raw_dir, ignore_errors=True)
    print(f"  frames: {raw_count} scene cuts -> {len(final)} kept")
    return final


## Notes files

Each video gets `notes.html` and `notes.docx`. The HTML page links each block to that moment on YouTube. When the video has chapters, they become collapsible sections in the HTML page and headings in the Word file, with a chapter list at the top of both. A playlist also gets `index.html` and one combined Word file.


In [24]:
def watch_url(video_id: str, moment: float) -> str:
    return f"https://www.youtube.com/watch?v={video_id}&t={max(0, int(moment))}s"


def add_hyperlink(paragraph, text: str, url: str) -> None:
    relation_id = paragraph.part.relate_to(url, HYPERLINK_REL, is_external=True)
    hyperlink = OxmlElement("w:hyperlink")
    hyperlink.set(qn("r:id"), relation_id)
    run = OxmlElement("w:r")
    props = OxmlElement("w:rPr")
    color = OxmlElement("w:color")
    color.set(qn("w:val"), "1155CC")
    underline = OxmlElement("w:u")
    underline.set(qn("w:val"), "single")
    size = OxmlElement("w:sz")
    size.set(qn("w:val"), "20")
    props.extend([color, underline, size])
    run.append(props)
    node = OxmlElement("w:t")
    node.text = text
    run.append(node)
    hyperlink.append(run)
    paragraph._p.append(hyperlink)


def add_internal_link(paragraph, text: str, anchor: str) -> None:
    hyperlink = OxmlElement("w:hyperlink")
    hyperlink.set(qn("w:anchor"), anchor)
    run = OxmlElement("w:r")
    props = OxmlElement("w:rPr")
    color = OxmlElement("w:color")
    color.set(qn("w:val"), "1155CC")
    underline = OxmlElement("w:u")
    underline.set(qn("w:val"), "single")
    props.extend([color, underline])
    run.append(props)
    node = OxmlElement("w:t")
    node.text = text
    run.append(node)
    hyperlink.append(run)
    paragraph._p.append(hyperlink)


def add_bookmark(paragraph, name: str, bookmark_id: int) -> None:
    start = OxmlElement("w:bookmarkStart")
    start.set(qn("w:id"), str(bookmark_id))
    start.set(qn("w:name"), name)
    end = OxmlElement("w:bookmarkEnd")
    end.set(qn("w:id"), str(bookmark_id))
    paragraph._p.insert(0, start)
    paragraph._p.append(end)


def group_by_chapter(segments: list[Segment]) -> list[tuple[str | None, list[Segment]]]:
    groups: list[tuple[str | None, list[Segment]]] = []
    current: str | None = None
    bucket: list[Segment] = []
    started = False
    for segment in segments:
        if started and segment.chapter != current:
            groups.append((current, bucket))
            bucket = []
        current = segment.chapter
        bucket.append(segment)
        started = True
    if bucket:
        groups.append((current, bucket))
    return groups


def configure_doc(document: Document) -> None:
    section = document.sections[0]
    section.left_margin = Inches(0.75)
    section.right_margin = Inches(0.75)
    section.top_margin = Inches(0.7)
    section.bottom_margin = Inches(0.7)
    normal = document.styles["Normal"]
    normal.font.name = "Calibri"
    normal.font.size = Pt(12)
    normal.paragraph_format.space_after = Pt(8)


def _add_text_paragraphs(document: Document, text: str) -> None:
    for paragraph in [part.strip() for part in text.split("\n\n") if part.strip()]:
        document.add_paragraph(paragraph)


def add_video_body(document: Document, title: str, video_id: str, source_url: str, segments: list[Segment], settings: Settings) -> None:
    heading = document.add_heading(title, level=1)
    heading.alignment = 1
    source = document.add_paragraph()
    add_hyperlink(source, "Source video", source_url)
    groups = group_by_chapter(segments)
    named = [(name, items) for name, items in groups if name]
    if named:
        label = document.add_paragraph()
        label_run = label.add_run("Chapters")
        label_run.bold = True
        for number, (name, items) in enumerate(named, start=1):
            line = document.add_paragraph()
            add_internal_link(line, f"{fmt_ts(items[0].start)}  {name}", f"chapter_{number}")
    chapter_number = 0
    for name, items in groups:
        if name:
            chapter_number += 1
            chapter_heading = document.add_heading(name, level=2)
            add_bookmark(chapter_heading, f"chapter_{chapter_number}", chapter_number)
        for segment in items:
            clock = document.add_paragraph()
            stamp = clock.add_run(f"{fmt_ts(segment.start)} – {fmt_ts(segment.end)}  ")
            stamp.italic = True
            stamp.font.size = Pt(10)
            stamp.font.color.rgb = RGBColor(0x66, 0x66, 0x66)
            add_hyperlink(clock, "Watch this moment", watch_url(video_id, segment.start))
            if settings.text_then_image:
                if segment.text:
                    _add_text_paragraphs(document, segment.text)
                if segment.image.exists():
                    document.add_picture(str(segment.image), width=Inches(6.5))
            else:
                if segment.image.exists():
                    document.add_picture(str(segment.image), width=Inches(6.5))
                if segment.text:
                    _add_text_paragraphs(document, segment.text)


def write_docx(path: Path, title: str, video_id: str, source_url: str, segments: list[Segment], settings: Settings) -> None:
    document = Document()
    configure_doc(document)
    document.core_properties.title = title
    add_video_body(document, title, video_id, source_url, segments, settings)
    path.parent.mkdir(parents=True, exist_ok=True)
    document.save(path)


def _paragraphs_html(text: str) -> str:
    parts = [part.strip() for part in text.split("\n\n") if part.strip()]
    return "".join(f"<p>{html.escape(part)}</p>" for part in parts)


def _embedded_image(path: Path, alt: str) -> str:
    """Inline the JPEG so the HTML file shows the frame without the frames folder."""
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f'<img src="data:image/jpeg;base64,{encoded}" alt="{html.escape(alt)}">'


def _segment_html(segment: Segment, video_id: str, settings: Settings) -> str:
    image = _embedded_image(segment.image, f"Frame at {fmt_ts(segment.start)}") if segment.image.exists() else ""
    text = _paragraphs_html(segment.text)
    body = f"{text}{image}" if settings.text_then_image else f"{image}{text}"
    return (
        '<section class="segment">'
        f'<p class="when">{fmt_ts(segment.start)} – {fmt_ts(segment.end)} '
        f'<a href="{html.escape(watch_url(video_id, segment.start))}">Watch this moment</a></p>'
        f"{body}</section>"
    )


def write_html(path: Path, title: str, video_id: str, source_url: str, segments: list[Segment], subtitle_label: str, settings: Settings) -> None:
    groups = group_by_chapter(segments)
    has_chapters = any(name for name, _items in groups)
    blocks = []
    nav_items = []
    chapter_number = 0
    for name, items in groups:
        body = "".join(_segment_html(segment, video_id, settings) for segment in items)
        if has_chapters and name:
            chapter_number += 1
            anchor = f"chapter-{chapter_number}"
            stamp = fmt_ts(items[0].start)
            nav_items.append(f'<li><a href="#{anchor}">{html.escape(stamp)} · {html.escape(name)}</a></li>')
            blocks.append(
                f'<details class="chapter" id="{anchor}" open>'
                f"<summary>{html.escape(name)} <span>{html.escape(stamp)}</span></summary>"
                f"{body}</details>"
            )
        else:
            blocks.append(body)
    nav = ""
    if nav_items:
        nav = '<nav class="chapters"><p class="meta">Chapters</p><ol>' + "".join(nav_items) + "</ol></nav>"
    page = f"""<!DOCTYPE html>
<html lang=\"en\">
<head>
<meta charset=\"utf-8\">
<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">
<title>{html.escape(title)}</title>
<style>
body {{ font-family: Georgia, "Iowan Old Style", serif; max-width: 860px; margin: 2rem auto; padding: 0 1.25rem 3rem; color: #1c1c1c; line-height: 1.55; }}
h1 {{ font-size: 1.7rem; line-height: 1.25; }}
.meta {{ color: #555; font-family: "Segoe UI", sans-serif; font-size: 0.92rem; }}
.chapters ol {{ padding-left: 1.2rem; }}
.chapters li {{ margin: 0.25rem 0; }}
details.chapter {{ margin: 1.6rem 0 2rem; border-top: 1px solid #e4e4e4; padding-top: 0.75rem; }}
details.chapter summary {{ cursor: pointer; font-size: 1.15rem; font-weight: 600; }}
details.chapter summary span {{ font-weight: 400; color: #666; font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 0.82rem; margin-left: 0.4rem; }}
.segment {{ margin: 1.4rem 0 2rem; }}
.when {{ font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 0.82rem; color: #666; margin-bottom: 0.4rem; }}
img {{ width: 100%; height: auto; border: 1px solid #e4e4e4; border-radius: 6px; }}
a {{ color: #1558b0; }}
</style>
</head>
<body>
<h1>{html.escape(title)}</h1>
<p class=\"meta\"><a href=\"{html.escape(source_url)}\">Source video</a>
 · subtitles: {html.escape(subtitle_label)}</p>
{nav}
{''.join(blocks)}
</body>
</html>
"""
    path.write_text(page, encoding="utf-8")


def write_index(path: Path, rows: list[dict]) -> None:
    items = []
    for row in rows:
        if row.get("error"):
            items.append(f"<li>{html.escape(row.get('title') or row.get('url') or 'Video')} — failed: {html.escape(row['error'])}</li>")
            continue
        html_rel = Path(row["html"]).relative_to(path.parent).as_posix()
        docx_rel = Path(row["docx"]).relative_to(path.parent).as_posix()
        items.append(
            "<li>"
            f"<a href=\"{html.escape(html_rel)}\">{html.escape(row['title'])}</a> "
            f"(<a href=\"{html.escape(docx_rel)}\">Word</a>)"
            "</li>"
        )
    document = f"""<!DOCTYPE html>
<html lang=\"en\">
<head>
<meta charset=\"utf-8\">
<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">
<title>Lecture notes</title>
<style>
body {{ font-family: "Segoe UI", sans-serif; max-width: 720px; margin: 2rem auto; padding: 0 1.25rem; line-height: 1.5; }}
a {{ color: #1558b0; }}
</style>
</head>
<body>
<h1>Lecture notes</h1>
<ul>
{''.join(items)}
</ul>
</body>
</html>
"""
    path.write_text(document, encoding="utf-8")


def slugify(title: str, video_id: str) -> str:
    cleaned = re.sub(r"[^\w.-]+", "_", title, flags=re.UNICODE)
    cleaned = re.sub(r"_+", "_", cleaned).strip("._")[:70].strip("_")
    if not cleaned:
        return video_id
    return f"{cleaned}_{video_id}"


## Pipeline


In [25]:
def _video_path(folder: Path, video_id: str) -> Path | None:
    for path in folder.iterdir():
        if path.stem == video_id and path.suffix.lower() in {".mp4", ".webm", ".mkv", ".mov"}:
            return path
    return None


def _subtitle_path(folder: Path, video_id: str, info: dict) -> tuple[Path | None, str]:
    requested = info.get("requested_subtitles") or {}
    ranked: list[tuple[int, Path, str]] = []
    for meta in requested.values():
        filepath = meta.get("filepath")
        if not filepath:
            continue
        path = Path(filepath)
        if not path.is_absolute():
            path = folder / path
        if not path.exists() or "live_chat" in path.name:
            continue
        label = str(meta.get("name") or path.stem)
        auto_rank = 1 if "auto" in label.lower() else 0
        ranked.append((auto_rank, path, label))
    if ranked:
        ranked.sort(key=lambda item: (item[0], len(item[1].name)))
        _, path, label = ranked[0]
        return path, label
    files = [path for path in folder.glob(f"{video_id}*.vtt") if "live_chat" not in path.name]
    if not files:
        return None, "none"
    files.sort(key=lambda path: (".auto." in path.name, len(path.name)))
    return files[0], files[0].name


def process_video(url: str, output_dir: Path, settings: Settings) -> dict:
    work = output_dir / "_download"
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True)
    print(f"  downloading {url}")
    info = download_video(url, work, settings)
    info_file = next(work.glob("*.info.json"), None)
    if info_file:
        saved = json.loads(info_file.read_text(encoding="utf-8"))
        # The info file has chapters and duration. Keep caption URLs if either copy has them.
        for key in ("subtitles", "automatic_captions"):
            if not saved.get(key) and info.get(key):
                saved[key] = info[key]
        info = saved
    caption_path, subtitle_label = save_captions(info, work, settings)
    video_id = str(info.get("id") or "")
    title = str(info.get("title") or video_id or url)
    if not video_id:
        raise RuntimeError("Download finished without a video id")
    video = _video_path(work, video_id)
    if video is None:
        raise RuntimeError("Download finished without a video file")
    duration = float(info.get("duration") or 0) or probe_duration(video)
    if duration <= 0:
        duration = probe_duration(video)
    if caption_path is None:
        caption_path, subtitle_label = _subtitle_path(work, video_id, info)
    if caption_path is None:
        cues: list[Cue] = []
        print("  no subtitles found; notes will be frames only")
    else:
        raw_captions = caption_path.read_text(encoding="utf-8", errors="replace")
        parsed = parse_json3_text(raw_captions) if caption_path.suffix == ".json3" else parse_vtt_text(raw_captions)
        cues = prepare_cues(parsed)
        kind = "pause punctuation" if captions_lack_punctuation(cues) else "original punctuation"
        print(f"  subtitles: {subtitle_label} -> {len(cues)} cues ({kind})")
    slug = slugify(title, video_id)
    video_dir = output_dir / slug
    frame_dir = video_dir / "frames"
    if frame_dir.exists():
        shutil.rmtree(frame_dir)
    frames = extract_frames(video, frame_dir, settings, duration)
    segments = build_segments(frames, cues, duration, info.get("chapters"), settings)
    source_url = str(info.get("webpage_url") or url)
    html_path = video_dir / "notes.html"
    docx_path = video_dir / "notes.docx"
    write_html(html_path, title, video_id, source_url, segments, subtitle_label, settings)
    write_docx(docx_path, title, video_id, source_url, segments, settings)
    spoken = sum(1 for segment in segments if segment.text)
    print(f"  notes: {len(segments)} blocks, {spoken} with speech")
    print(f"  {html_path}")
    print(f"  {docx_path}")
    if settings.keep_video:
        kept = video_dir / video.name
        shutil.move(str(video), kept)
    shutil.rmtree(work, ignore_errors=True)
    return {
        "title": title,
        "video_id": video_id,
        "url": source_url,
        "html": html_path,
        "docx": docx_path,
        "segments": segments,
    }


def build_lecture_notes(
    source_url: str,
    output_dir: str | Path = "output",
    playlist_limit: int | None = None,
    settings: Settings | None = None,
) -> list[dict]:
    settings = settings or Settings()
    settings.validate()
    source_url = source_url.strip()
    if not source_url:
        raise ValueError("Paste a YouTube video or playlist URL into SOURCE_URL")
    root = Path(output_dir).expanduser().resolve()
    root.mkdir(parents=True, exist_ok=True)
    sources = list_sources(source_url, playlist_limit, settings)
    if not sources:
        raise RuntimeError("No videos found at that URL")
    print(f"{len(sources)} video(s) -> {root}")
    results: list[dict] = []
    combined: Document | None = None
    written = 0
    for index, source in enumerate(sources, start=1):
        title = source.get("title") or source.get("url")
        print(f"[{index}/{len(sources)}] {title}")
        try:
            result = process_video(source["url"], root, settings)
        except Exception as exc:
            message = str(exc).strip() or exc.__class__.__name__
            print(f"  failed: {message}")
            results.append({"title": title, "url": source.get("url"), "error": message})
            continue
        results.append(result)
        if combined is None:
            combined = Document()
            configure_doc(combined)
            if len(sources) > 1:
                combined.add_heading("Lecture notes", level=0)
        elif len(sources) > 1:
            combined.add_page_break()
        if len(sources) > 1 and combined is not None:
            add_video_body(combined, result["title"], result["video_id"], result["url"], result["segments"], settings)
            written += 1
    if combined is not None and len(sources) > 1 and written:
        combined_path = root / "lecture_notes.docx"
        combined.save(combined_path)
        print(f"Combined Word file: {combined_path}")
    if results:
        write_index(root / "index.html", results)
        print(f"Index: {root / 'index.html'}")
    return results


def preview_notes(results: list[dict], limit: int = 3) -> None:
    shown = next((row for row in results if row.get("segments")), None)
    if shown is None:
        return
    try:
        from IPython.display import Image as IPyImage
        from IPython.display import Markdown, display
    except Exception:
        return
    display(Markdown(f"### {shown['title']}"))
    for segment in shown["segments"][:limit]:
        clock = f"`{fmt_ts(segment.start)} – {fmt_ts(segment.end)}`"
        if segment.text:
            display(Markdown(segment.text))
        if segment.image.exists():
            display(IPyImage(filename=str(segment.image)))
        display(Markdown(clock))


## Check the clock

This cell does not touch YouTube. Repeated caption words are dropped, and the words spoken between two frames are placed above the later frame.


In [ ]:
_sample_vtt = """WEBVTT

00:00:00.000 --> 00:00:01.500
hello there

00:00:00.500 --> 00:00:02.400
hello there friends

00:00:04.000 --> 00:00:06.000
welcome to the lecture

00:00:05.200 --> 00:00:07.500
welcome to the lecture today
"""

_cues = prepare_cues(parse_vtt_text(_sample_vtt))
assert [(round(cue.start, 1), round(cue.end, 1), cue.text) for cue in _cues] == [
    (0.0, 1.5, "hello there"),
    (0.5, 2.4, "friends"),
    (4.0, 6.0, "welcome to the lecture"),
    (5.2, 7.5, "today"),
]

_frames = [
    FrameShot(0.0, Path("slide_a.jpg")),
    FrameShot(3.0, Path("slide_b.jpg")),
    FrameShot(6.0, Path("slide_c.jpg")),
]
_segments = build_segments(_frames, _cues, 8.0, None, Settings())
assert _segments[0].text == ""
assert _segments[0].image.name == "slide_a.jpg"
assert _segments[1].text == "hello there friends"
assert _segments[1].image.name == "slide_b.jpg"
assert abs(_segments[1].start - 0.0) < 1e-6 and abs(_segments[1].end - 3.0) < 1e-6
assert _segments[2].text == "welcome to the lecture today"
assert _segments[2].image.name == "slide_c.jpg"

_cut = [
    Cue(0.0, 2.0, "We update the model parameters at"),
    Cue(2.0, 3.2, "each step."),
    Cue(4.5, 6.0, "Then we continue."),
]
_cut_frames = [
    FrameShot(0.0, Path("a.jpg")),
    FrameShot(2.0, Path("b.jpg")),
    FrameShot(6.0, Path("c.jpg")),
]
_cut_segments = build_segments(_cut_frames, _cut, 6.0, None, Settings())
assert _cut_segments[1].text == "We update the model parameters at each step."
assert _cut_segments[1].image.name == "b.jpg"
assert _cut_segments[2].text == "Then we continue."
print("Sync check passed.")
for _segment in _segments:
    print(f"  {fmt_ts(_segment.start)} – {fmt_ts(_segment.end)} | {_segment.image.name} | {_segment.text}")


## Run

Paste a watch URL or a playlist URL. `PLAYLIST_LIMIT` caps how many playlist items are processed (`None` means the whole playlist). Output is written under `OUTPUT_DIR`, relative to the folder where Jupyter was started.


In [ ]:
# --- edit these ---
SOURCE_URL = ""  # https://www.youtube.com/watch?v=...  or a /playlist?list=... URL
OUTPUT_DIR = "output"
PLAYLIST_LIMIT = None  # e.g. 3 while trying a long playlist

settings = Settings(
    max_height=720,
    scene_threshold=0.30,
    min_scene_gap=1.0,
    max_gap=40.0,
    duplicate_threshold=6.0,
    paragraph_pause=1.15,
    text_then_image=True,
    keep_video=False,
    sub_langs=("en",),
    cookies_from_browser=None,  # same machine only. Safari works on macOS, not on Linux/Colab.
    cookies_file=None,  # Netscape cookies.txt uploaded into this runtime. Use this on Linux/Colab.
)

if not SOURCE_URL.strip():
    print("Set SOURCE_URL, then run this cell again.")
else:
    results = build_lecture_notes(
        SOURCE_URL,
        output_dir=OUTPUT_DIR,
        playlist_limit=PLAYLIST_LIMIT,
        settings=settings,
    )
    preview_notes(results, limit=3)


## Tuning

- Missed slide changes: lower `scene_threshold` to `0.15` or `0.20`.
- Too many near-identical frames: raise `duplicate_threshold` (try `12`) or `min_scene_gap`.
- A static camera produces one huge text block: lower `max_gap`.
- Notes feel late or early by a caption: the words are cut on the frame times. Speech between frame 9 and frame 10 is printed above frame 10. Scene cuts are exact; gap-fill frames are fast seeks and can sit on the nearest keyframe.
- Another spoken language: set `sub_langs` to `("hi",)` or `("en",)`. A manual track in that language is used when YouTube has one; otherwise the auto-captions are used.
- HTTP 429 on captions means YouTube is refusing the timed-text request for a while. The notebook waits and retries. On Linux or Colab, export a Netscape `cookies.txt` from the browser where you are logged into YouTube, upload it, and set `cookies_file` to that path. Leave `cookies_from_browser` as `None`. Safari cookies can be read only when the kernel itself is running on macOS.
- The HTML file contains the frames itself, so the pictures show when you open or download `notes.html`. The `frames` folder is still written next to it.
